# Combining All Datasets into a Single Dataset

## Importing Libraries

In [1]:
import pandas as pd

## Importing Datasets

In [2]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
product_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

## Merging Datasets with GeoLocation

In [3]:
geo_avg = (
    geolocation.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]]
    .mean()
    .reset_index()
)

sellers_inner = pd.merge(
    sellers,
    geo_avg,
    left_on="seller_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="inner"
).drop(columns=["geolocation_zip_code_prefix"]).rename(columns={
    "geolocation_lat": "seller_lat",
    "geolocation_lng": "seller_lng"
})

customers_inner = pd.merge(
    customers,
    geo_avg,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="inner"
).drop(columns=["geolocation_zip_code_prefix"]).rename(columns={
    "geolocation_lat": "customer_lat",
    "geolocation_lng": "customer_lng"
})


df_geo = pd.merge(orders, customers_inner, on="customer_id", how="inner")
df_geo = pd.merge(df_geo, order_items, on="order_id", how="inner")
df_geo = pd.merge(df_geo, sellers_inner, on="seller_id", how="inner")
df_geo = pd.merge(df_geo, order_payments, on="order_id", how="inner")
df_geo = pd.merge(df_geo, order_reviews, on="order_id", how="inner")
df_geo = pd.merge(df_geo, products, on="product_id", how="inner")
df_geo = pd.merge(df_geo, product_translation, on="product_category_name", how="inner")

df_geo.shape

(115037, 44)

## Converting Dataset with GeoLocation to CSV

In [4]:
df_geo.to_csv("../data/raw/olist_geocombined_dataset.csv", index=False)

## Merging Datasets without GeoLocation

In [5]:
df = pd.merge(orders, customers, on="customer_id", how="left")
df = pd.merge(df, order_payments, on="order_id", how="left")
df = pd.merge(df, order_reviews, on="order_id", how="left")
df = pd.merge(df, order_items, on="order_id", how="left")
df = pd.merge(df, products, on="product_id", how="left")
df = pd.merge(df, sellers, on="seller_id", how="left")
df = pd.merge(df, product_translation, on="product_category_name", how="left")

## Converting Dataset to CSV

In [6]:
df.to_csv("../data/raw/olist_combined_dataset.csv", index=False)